# 第2回 宿題・基礎：エポック数20と10を比較する

元のFAQ演習のコピーです。変更する学習条件は `num_train_epochs: 20 → 10` の1つだけです。モデル・FAQ・質問・正解・プロンプト・学習率・LoRAランク・生成上限96・生成設定・採点方法は維持します。出力先のみ別にしています。

**新規または再起動したT4ランタイムで、0〜8を上から実行してください。** 既存の学習済みモデルから追加学習せず、元のモデルを読み直します。4の再実行は避けてください。追加実行したい場合はランタイムを再起動してください。

8で元の20エポックの結果と比較し、バックアップ出力を生成します。最後に⌘Sでノートブックを保存してください。比較元の結果・ログは埋め込み済みでアップロード不要です。

形式・内容の採点は元演習の定義のままです。内容評価は意味の正しさ全般を保証しません。生成条件は変更せず、JSONとして完結しているかも比較時に表示します。


## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 10` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 20 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 10,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 20 ステップ
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_basic_epochs10",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_basic_epochs10",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
BASELINE = json.loads('{"config": {"base_model": "Qwen/Qwen3.5-0.8B", "revision": "2fc06364715b967f1860aea9cf38778875588b17", "seed": 42, "num_train_epochs": 20, "warmup_steps": 6, "lr": 0.0002, "lora_r": 16, "lora_alpha": 32, "max_seq_len": 512, "max_new_tokens": 96, "gen": {"do_sample": false}}, "versions": {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "torch": "2.11.0+cu130"}, "method": "LoRA（量子化なし）", "before": {"label": "before", "n": 21, "format_rate": 0.09523809523809523, "content_rate": 0.0, "both_rate": 0.0, "seconds": 22.0}, "after": {"label": "after", "n": 21, "format_rate": 1.0, "content_rate": 0.9047619047619048, "both_rate": 0.9047619047619048, "seconds": 42.4}, "regressed": [], "knowledge": [{"質問": "この講座の最終課題の締切はいつですか", "FAQ にある": false, "学習前の分類": "format_error", "学習前の出力": "{\\"answer\\": \\"9 月 30 日 23:59\\"}", "学習後の分類": "asserted", "学習後の出力": "{\\"answer\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}"}, {"質問": "担当教員の名前は何ですか", "FAQ にある": false, "学習前の分類": "format_error", "学習前の出力": "{\\"answer\\": \\"確認が必要\\"}", "学習後の分類": "needs_review", "学習後の出力": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}"}, {"質問": "次回の講義のテーマは何ですか", "FAQ にある": false, "学習前の分類": "format_error", "学習前の出力": "{\\"answer\\": \\"次回の講義のテーマは「学習ポータル」の「課題」欄で提出する PDF 1 ファイルを提出します。\\"}", "学習後の分類": "needs_review", "学習後の出力": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}"}, {"質問": "課題の提出期限はいつですか", "FAQ にある": false, "学習前の分類": "format_error", "学習前の出力": "{\\"answer\\": \\"9 月 30 日 23:59\\"}", "学習後の分類": "asserted", "学習後の出力": "{\\"answer\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}"}], "executed_training_myself": true, "training": {"seconds": 252, "seconds_precision": "元ノートブック表示の秒単位の丸め値", "final_logged_loss": 0.013854, "final_logged_step": 40, "loss_source": "元ノートブックのTraining Loss表（表示精度6桁、直近ログ区間の平均）"}}')
EXPECTED_TEST = json.loads('[{"question": "提出の期限は？", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "提出期限の日付と時刻を教えてください", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "課題の締切は何月何日の何時ですか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "提出場所は？", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"提出先は学習ポータルの「課題」欄です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "課題の提出はどこで行いますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"提出先は学習ポータルの「課題」欄です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "提出物を出す先を教えてください", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"提出先は学習ポータルの「課題」欄です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "PDF 以外でも出せますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "提出するファイルの形式は何ですか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "提出ファイルは何形式で、いくつ出しますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "相談できる時間帯は？", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "オフィスアワーの曜日と時間を教えてください", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "質問を受け付けてもらえるのは何曜日の何時からですか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "締切後の扱いを教えてください", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "提出が締切に間に合わなかったらどうなりますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "期限後に提出した課題は受理されますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。\\", \\"needs_review\\": false}", "needs_review": false}, {"question": "成績はいつ出ますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}", "needs_review": true}, {"question": "講義の録画は公開されますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}", "needs_review": true}, {"question": "課題の文字数に制限はありますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}", "needs_review": true}, {"question": "提出した課題へのフィードバックはもらえますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}", "needs_review": true}, {"question": "参考文献の書き方に指定はありますか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}", "needs_review": true}, {"question": "オフィスアワーは事前予約が必要ですか", "faq": "- 課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\n- 提出先は学習ポータルの「課題」欄です。\\n- 提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\n- 質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\n- 期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。", "target": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}", "needs_review": true}]')
EXPECTED_GOLD = json.loads('{"提出の期限は？": {"needs_review": false, "must_include": ["9 月 30 日", "23:59"]}, "提出期限の日付と時刻を教えてください": {"needs_review": false, "must_include": ["9 月 30 日", "23:59"]}, "課題の締切は何月何日の何時ですか": {"needs_review": false, "must_include": ["9 月 30 日", "23:59"]}, "提出場所は？": {"needs_review": false, "must_include": ["学習ポータル", "課題"]}, "課題の提出はどこで行いますか": {"needs_review": false, "must_include": ["学習ポータル", "課題"]}, "提出物を出す先を教えてください": {"needs_review": false, "must_include": ["学習ポータル", "課題"]}, "PDF 以外でも出せますか": {"needs_review": false, "must_include": ["PDF"]}, "提出するファイルの形式は何ですか": {"needs_review": false, "must_include": ["PDF"]}, "提出ファイルは何形式で、いくつ出しますか": {"needs_review": false, "must_include": ["PDF"]}, "相談できる時間帯は？": {"needs_review": false, "must_include": ["火曜", "17:00"]}, "オフィスアワーの曜日と時間を教えてください": {"needs_review": false, "must_include": ["火曜", "17:00"]}, "質問を受け付けてもらえるのは何曜日の何時からですか": {"needs_review": false, "must_include": ["火曜", "17:00"]}, "締切後の扱いを教えてください": {"needs_review": false, "must_include": ["48 時間", "減点"]}, "提出が締切に間に合わなかったらどうなりますか": {"needs_review": false, "must_include": ["48 時間", "減点"]}, "期限後に提出した課題は受理されますか": {"needs_review": false, "must_include": ["48 時間", "減点"]}, "成績はいつ出ますか": {"needs_review": true, "must_include": ["確認"]}, "講義の録画は公開されますか": {"needs_review": true, "must_include": ["確認"]}, "課題の文字数に制限はありますか": {"needs_review": true, "must_include": ["確認"]}, "提出した課題へのフィードバックはもらえますか": {"needs_review": true, "must_include": ["確認"]}, "参考文献の書き方に指定はありますか": {"needs_review": true, "must_include": ["確認"]}, "オフィスアワーは事前予約が必要ですか": {"needs_review": true, "must_include": ["確認"]}}')

current = {k:v for k,v in CONFIG.items() if k not in ("work_dir", "drive_dir")}
config_diff = {k: (BASELINE["config"].get(k), current.get(k)) for k in set(BASELINE["config"]) | set(current) if BASELINE["config"].get(k) != current.get(k)}
assert config_diff == {"num_train_epochs": (20, 10)}, config_diff
current_versions = {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]}
assert current_versions == BASELINE["versions"], f"ライブラリ版が比較元と異なります: {current_versions}"
assert "Tesla T4" in gpu_info, "比較元と同じT4を選択してください"
print("比較元からのCONFIG差分:", config_diff)


{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 10,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_basic_epochs10",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_basic_epochs10"
}
比較元からのCONFIG差分: {'num_train_epochs': (20, 10)}


## 2. 架空 FAQ と教師データの生成

**やること**: 5 項目の架空 FAQ から、学習用（train）、検証用（valid）、評価用（test）のデータを作ります。

モデルへの入力は「FAQ ＋ 質問」、望ましい出力は `{"answer": <文字列>, "needs_review": <真偽値>}` の JSON 1 行です。

- FAQ に根拠がある質問には、FAQ の文で答え、`needs_review: false` にする。
- FAQ に無い質問には、「この FAQ には記載がありません。運営に確認が必要です。」と答え、`needs_review: true` にする。

**データの分け方**

- 同じ FAQ 項目を聞く質問の言い換えを、**テンプレート単位**で train / valid / test に振り分けます。学習で見た文がそのまま test に出ないようにするためです。
- test は 21 問（根拠あり 15、根拠なし 6）です。件数を確保するため、test にだけ使う質問を別に用意しています。
- 正解（`needs_review` の値と、答えに含まれるべき語句）は `gold` に分けて持ち、モデルには渡しません。
- train は 14 件（根拠あり 10、根拠なし 4）、valid は 7 件です。FAQ が 5 項目しかないので学習データは少なく、同じ例を何回も学習することになります。何回学習するかは、データを複製して増やすのではなく、セクション 4 のエポック数（`num_train_epochs`）で決めます。データのファイルには実例だけを入れ、学習量は学習の設定で持つのが基本です。

**見るところ**: `train=14 valid=7 test=21 gold=21` と、学習例の 1 件（質問と、望ましい出力）が表示されます。質問が train / valid / test のどれかで重複していると、ここでエラーになります。

In [4]:
FAQ = {
    "deadline": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
    "destination": "提出先は学習ポータルの「課題」欄です。",
    "format": "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
    "office_hours": "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
    "late": "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# 根拠あり: (FAQ キー, 質問テンプレート群, 正解の answer に含まれるべき語)
GROUNDED = {
    "deadline": (["課題の提出期限はいつですか", "いつまでに出せばいいですか", "締切を教えてください", "提出の期限は？"], ["9 月 30 日", "23:59"]),
    "destination": (["提出先はどこですか", "どこにアップロードしますか", "課題はどこから出しますか", "提出場所は？"], ["学習ポータル", "課題"]),
    "format": (["提出形式を教えてください", "ファイル名の決まりはありますか", "何のファイル形式で出しますか", "PDF 以外でも出せますか"], ["PDF"]),
    "office_hours": (["質問はいつできますか", "オフィスアワーはいつですか", "先生に質問する時間はありますか", "相談できる時間帯は？"], ["火曜", "17:00"]),
    "late": (["遅れて出したらどうなりますか", "期限を過ぎた場合は受け付けますか", "遅延提出のルールは？", "締切後の扱いを教えてください"], ["48 時間", "減点"]),
}
# 根拠なし: FAQ に無いので needs_review=true が正解
UNGROUNDED = [
    "再提出はできますか", "欠席した回の扱いを教えてください", "他の受講者の点数は見られますか",
    "成績はいつ出ますか", "提出期限は延長できますか", "グループでの提出は可能ですか",
    "課題の配点はどうなっていますか", "講義の録画は公開されますか",
]

# test 専用の追加質問（train / valid には入れない）
TEST_EXTRA_GROUNDED = {
    "deadline": ["提出期限の日付と時刻を教えてください", "課題の締切は何月何日の何時ですか"],
    "destination": ["課題の提出はどこで行いますか", "提出物を出す先を教えてください"],
    "format": ["提出するファイルの形式は何ですか", "提出ファイルは何形式で、いくつ出しますか"],
    "office_hours": ["オフィスアワーの曜日と時間を教えてください", "質問を受け付けてもらえるのは何曜日の何時からですか"],
    "late": ["提出が締切に間に合わなかったらどうなりますか", "期限後に提出した課題は受理されますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "課題の文字数に制限はありますか", "提出した課題へのフィードバックはもらえますか",
    "参考文献の書き方に指定はありますか", "オフィスアワーは事前予約が必要ですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")
assert test == EXPECTED_TEST, "評価用の入力データが比較元と異なります"
assert gold == EXPECTED_GOLD, "評価用の正解が比較元と異なります"
print("比較元と評価データ・正解が一致しています")


train=14 valid=7 test=21 gold=21
学習例の 1 件 | 質問: 課題の提出期限はいつですか | 望ましい出力: {"answer": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。", "needs_review": false}
比較元と評価データ・正解が一致しています


## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_basic_epochs10/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 21, "format_rate": 0.09523809523809523, "content_rate": 0.0, "both_rate": 0.0, "seconds": 21.8}


,質問,正解の needs_review,学習前の出力,形式,形式の失敗理由,内容
0,提出の期限は？,False,"{""answer"": ""9 月 30 日 23:59""}",×,missing_key,×
1,提出期限の日付と時刻を教えてください,False,"{""answer"": ""9 月 30 日 23:59""}",×,missing_key,×
2,課題の締切は何月何日の何時ですか,False,"{""answer"": ""9 月 30 日 23:59""}",×,missing_key,×
3,提出場所は？,False,"{""answer"": ""学習ポータルの「課題」欄""}",×,missing_key,×
4,課題の提出はどこで行いますか,False,"{""answer"": ""学習ポータルの「課題」欄で提出先を指定し、PDF 1 ファイルで提出します。提出時間は毎週火曜 17:00〜18:00 のオフィスアワーで行います。""}",×,missing_key,×
5,提出物を出す先を教えてください,False,"{""answer"": ""学習ポータルの「課題」欄""}",×,missing_key,×
6,PDF 以外でも出せますか,False,"{""answer"": ""確認が必要""}",×,missing_key,×
7,提出するファイルの形式は何ですか,False,"{""answer"": ""PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です""}",×,missing_key,×
8,提出ファイルは何形式で、いくつ出しますか,False,"{""answer"": ""PDF 1 ファイルで、1 出します""}",×,missing_key,×
9,相談できる時間帯は？,False,"{""answer"": ""火曜 17:00〜18:00 のオフィスアワー""}",×,missing_key,×


## 4. LoRA 学習（10エポック・20ステップ）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

assert not getattr(model, "peft_config", None), "学習済みadapterが存在します。ランタイムを再起動して0から実行してください"
model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
train_output = trainer.train()
training_seconds = time.time()-t0
loss_logs = [r for r in trainer.state.log_history if "loss" in r]
assert loss_logs, "学習lossのログがありません"
final_logged_loss = loss_logs[-1]["loss"]
final_logged_step = loss_logs[-1]["step"]
print(f"学習時間: {training_seconds:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,1.568477
20,0.101157


学習時間: 114 秒 | ステップ数: 20 | エポック: 10.0


### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_basic_epochs10/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 40 ステップ（20 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。

学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。

In [8]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 21, "format_rate": 1.0, "content_rate": 0.8095238095238095, "both_rate": 0.8095238095238095, "seconds": 42.6}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,21,0.095238,0.000000,0.000000,21.8
after,21,1.000000,0.809524,0.809524,42.6


退行: []


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,提出の期限は？,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
1,提出期限の日付と時刻を教えてください,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
2,課題の締切は何月何日の何時ですか,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
3,提出場所は？,False,"{""answer"": ""学習ポータルの「課題」欄""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
4,課題の提出はどこで行いますか,False,"{""answer"": ""学習ポータルの「課題」欄で提出先を指定し、PDF 1 ファイルで提出します。提出時間は毎週火曜 17:00〜18:00 のオフィスアワーで行います。""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
5,提出物を出す先を教えてください,False,"{""answer"": ""学習ポータルの「課題」欄""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
6,PDF 以外でも出せますか,False,"{""answer"": ""確認が必要""}",×/×,"{""answer"": ""この FAQ には記載がありません。提出先は学習ポータルの「課題」欄です。"", ""needs_review"": true}",○/×
7,提出するファイルの形式は何ですか,False,"{""answer"": ""PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です""}",×/×,"{""answer"": ""提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。"", ""needs_review"": false}",○/○
8,提出ファイルは何形式で、いくつ出しますか,False,"{""answer"": ""PDF 1 ファイルで、1 出します""}",×/×,"{""answer"": ""提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。"", ""needs_review"": false}",○/○
9,相談できる時間帯は？,False,"{""answer"": ""火曜 17:00〜18:00 のオフィスアワー""}",×/×,"{""answer"": ""質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。"", ""needs_review"": false}",○/○


## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答を 3 つに分類します。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない（学習前に多い） |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

In [9]:
KNOWLEDGE_QUESTIONS = [
    "この講座の最終課題の締切はいつですか",       # FAQ に無い（架空）
    "担当教員の名前は何ですか",                     # FAQ に無い
    "次回の講義のテーマは何ですか",                 # FAQ に無い
    "課題の提出期限はいつですか",                   # FAQ にある（対照）
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "FAQ にある": q in gold, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                 "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

,質問,FAQ にある,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,この講座の最終課題の締切はいつですか,False,format_error,"{""answer"": ""9 月 30 日 23:59""}",asserted,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}"
1,担当教員の名前は何ですか,False,format_error,"{""answer"": ""確認が必要""}",needs_review,"{""answer"": ""この FAQ には記載がありません。必ず与えられた FAQ に確認が必要です。"", ""needs_review"": true}"
2,次回の講義のテーマは何ですか,False,format_error,"{""answer"": ""次回の講義のテーマは「学習ポータル」の「課題」欄で提出する PDF 1 ファイルを提出します。""}",needs_review,"{""answer"": ""この FAQ には記載がありません。必ず質問に答えてください。"", ""needs_review"": true}"
3,課題の提出期限はいつですか,False,format_error,"{""answer"": ""9 月 30 日 23:59""}",asserted,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}"


## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_basic_epochs10` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）
- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [10]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "training": {"seconds": training_seconds, "final_logged_loss": final_logged_loss, "final_logged_step": final_logged_step, "loss_note": "直近10ステップのログ区間の平均。全学習平均や最終1ステップのlossとは異なる"},
    "config_diff": config_diff,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 10,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）",
  "before": {
    "label": "before",
    "n": 21,
    "format_rate": 0.09523809523809523,
    "content_rate": 0.0,
    "both_rate": 0.0,
    "seconds": 21.8
  },
  "after": {
    "label": "after",
    "n": 21,
    "format_rate": 1.0,
    "content_rate": 0.8095238095238095,
    "both_rate": 0.8095238095238095,
    "seconds": 42.6
  },
  "regressed": [],
  "training": {
    "seconds": 114.12823510169983,
    "final_logged_loss": 0.10115675926208496,
    "final_logged_step": 20,
    "loss_note": "直近10ステップのログ区間

## 8. 基礎課題の比較とバックアップ

学習後の形式遵守率・内容正解率、最終ログのloss、学習時間を比較します。元のlossは保存済みTraining Loss表のstep 40（0.013854）から取得。元の学習時間は画面表示の252秒で、丸められています。変更後はstep 20の最終ログを使用します。最終lossは直近のログ区間の平均です。

追加のJSON読取チェックは、元の回答原文と今回の回答原文を同じ条件で検査する診断です。生成上限到達そのものは採点に加えません。JSONが読めても意味の正しさや上限到達の有無を保証しません。

学習前の出力が前回と一致するかも表示します。一致しない場合は前提条件を確認し、そのまま変更の効果と断定しないでください。seed・環境をそろえてもGPUで完全な再現は保証されません。

実行後に⌘Sで保存してください。バックアップは結果ファイルのみで、学習済みadapterは含みません。


In [11]:
BASELINE_AFTER_JSON = '{\n  "summary": {\n    "label": "after",\n    "n": 21,\n    "format_rate": 1.0,\n    "content_rate": 0.9047619047619048,\n    "both_rate": 0.9047619047619048,\n    "seconds": 42.4\n  },\n  "rows": [\n    {\n      "question": "提出の期限は？",\n      "raw": "{\\"answer\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出期限の日付と時刻を教えてください",\n      "raw": "{\\"answer\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "課題の締切は何月何日の何時ですか",\n      "raw": "{\\"answer\\": \\"課題の提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出場所は？",\n      "raw": "{\\"answer\\": \\"提出先は学習ポータルの「課題」欄です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "課題の提出はどこで行いますか",\n      "raw": "{\\"answer\\": \\"提出先は学習ポータルの「課題」欄です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出物を出す先を教えてください",\n      "raw": "{\\"answer\\": \\"提出先は学習ポータルの「課題」欄です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "PDF 以外でも出せますか",\n      "raw": "{\\"answer\\": \\"提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出するファイルの形式は何ですか",\n      "raw": "{\\"answer\\": \\"提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出ファイルは何形式で、いくつ出しますか",\n      "raw": "{\\"answer\\": \\"提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "相談できる時間帯は？",\n      "raw": "{\\"answer\\": \\"質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "オフィスアワーの曜日と時間を教えてください",\n      "raw": "{\\"answer\\": \\"質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "質問を受け付けてもらえるのは何曜日の何時からですか",\n      "raw": "{\\"answer\\": \\"質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "締切後の扱いを教えてください",\n      "raw": "{\\"answer\\": \\"期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出が締切に間に合わなかったらどうなりますか",\n      "raw": "{\\"answer\\": \\"期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "期限後に提出した課題は受理されますか",\n      "raw": "{\\"answer\\": \\"期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": false\n    },\n    {\n      "question": "成績はいつ出ますか",\n      "raw": "{\\"answer\\": \\"提出期限は 9 月 30 日 23:59（日本時間）です。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": true\n    },\n    {\n      "question": "講義の録画は公開されますか",\n      "raw": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": true\n    },\n    {\n      "question": "課題の文字数に制限はありますか",\n      "raw": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": true\n    },\n    {\n      "question": "提出した課題へのフィードバックはもらえますか",\n      "raw": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": true\n    },\n    {\n      "question": "参考文献の書き方に指定はありますか",\n      "raw": "{\\"answer\\": \\"この FAQ には記載がありません。運営に確認が必要です。\\", \\"needs_review\\": true}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": true,\n      "both_ok": true,\n      "gold_needs_review": true\n    },\n    {\n      "question": "オフィスアワーは事前予約が必要ですか",\n      "raw": "{\\"answer\\": \\"質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": true\n    }\n  ]\n}'
BASELINE_BEFORE = json.loads('{\n  "summary": {\n    "label": "before",\n    "n": 21,\n    "format_rate": 0.09523809523809523,\n    "content_rate": 0.0,\n    "both_rate": 0.0,\n    "seconds": 22.0\n  },\n  "rows": [\n    {\n      "question": "提出の期限は？",\n      "raw": "{\\"answer\\": \\"9 月 30 日 23:59\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出期限の日付と時刻を教えてください",\n      "raw": "{\\"answer\\": \\"9 月 30 日 23:59\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "課題の締切は何月何日の何時ですか",\n      "raw": "{\\"answer\\": \\"9 月 30 日 23:59\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出場所は？",\n      "raw": "{\\"answer\\": \\"学習ポータルの「課題」欄\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "課題の提出はどこで行いますか",\n      "raw": "{\\"answer\\": \\"学習ポータルの「課題」欄で提出先を指定し、PDF 1 ファイルで提出します。提出時間は毎週火曜 17:00〜18:00 のオフィスアワーで行います。\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出物を出す先を教えてください",\n      "raw": "{\\"answer\\": \\"学習ポータルの「課題」欄\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "PDF 以外でも出せますか",\n      "raw": "{\\"answer\\": \\"確認が必要\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出するファイルの形式は何ですか",\n      "raw": "{\\"answer\\": \\"PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出ファイルは何形式で、いくつ出しますか",\n      "raw": "{\\"answer\\": \\"PDF 1 ファイルで、1 出します\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "相談できる時間帯は？",\n      "raw": "{\\"answer\\": \\"火曜 17:00〜18:00 のオフィスアワー\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "オフィスアワーの曜日と時間を教えてください",\n      "raw": "{\\"answer\\": \\"火曜 17:00〜18:00\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "質問を受け付けてもらえるのは何曜日の何時からですか",\n      "raw": "{\\"answer\\": \\"火曜 17:00〜18:00\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "締切後の扱いを教えてください",\n      "raw": "{\\"answer\\": \\"期限後 48 時間以内なら減点のうえ受け付けます。\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "提出が締切に間に合わなかったらどうなりますか",\n      "raw": "{\\"answer\\": \\"期限から 48 時間以内なら減点のうえ受け付けます\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "期限後に提出した課題は受理されますか",\n      "raw": "{\\"answer\\": \\"確認が必要\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": false\n    },\n    {\n      "question": "成績はいつ出ますか",\n      "raw": "{\\"answer\\": \\"確認が必要\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": true\n    },\n    {\n      "question": "講義の録画は公開されますか",\n      "raw": "{\\"answer\\": \\"否\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": true\n    },\n    {\n      "question": "課題の文字数に制限はありますか",\n      "raw": "{\\"answer\\": \\"なし\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": true\n    },\n    {\n      "question": "提出した課題へのフィードバックはもらえますか",\n      "raw": "{\\"answer\\": \\"確認が必要\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": true\n    },\n    {\n      "question": "参考文献の書き方に指定はありますか",\n      "raw": "{\\"answer\\": \\"確認が必要\\"}",\n      "format_ok": false,\n      "format_reason": "missing_key",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": true\n    },\n    {\n      "question": "オフィスアワーは事前予約が必要ですか",\n      "raw": "{\\"answer\\": \\"否\\", \\"needs_review\\": false}",\n      "format_ok": true,\n      "format_reason": "",\n      "content_ok": false,\n      "both_ok": false,\n      "gold_needs_review": true\n    }\n  ]\n}')

comparison = pd.DataFrame([
    {"experiment": "baseline_epochs20", "epochs": BASELINE["config"]["num_train_epochs"], "format_rate": BASELINE["after"]["format_rate"], "content_rate": BASELINE["after"]["content_rate"], "final_logged_loss": BASELINE["training"]["final_logged_loss"], "training_seconds": BASELINE["training"]["seconds"]},
    {"experiment": "basic_epochs10", "epochs": CONFIG["num_train_epochs"], "format_rate": after_summary["format_rate"], "content_rate": after_summary["content_rate"], "final_logged_loss": final_logged_loss, "training_seconds": round(training_seconds, 3)},
])
display(comparison)

def json_readable(raw):
    try:
        return isinstance(json.loads(raw), dict)
    except (ValueError, TypeError):
        return False

baseline_after = json.loads(BASELINE_AFTER_JSON)
assert [r["question"] for r in baseline_after["rows"]] == [r["question"] for r in after_rows]
diagnostics = []
for label, rows in [("baseline_before", BASELINE_BEFORE["rows"]), ("baseline_after", baseline_after["rows"]), ("basic_before", before_rows), ("basic_after", after_rows)]:
    diagnostics.append({"label": label, "n": len(rows), "json_readable": sum(json_readable(r["raw"]) for r in rows)})
display(pd.DataFrame(diagnostics))
assert [r["question"] for r in BASELINE_BEFORE["rows"]] == [r["question"] for r in before_rows]
initial_output_differences = [r["question"] for b,r in zip(BASELINE_BEFORE["rows"], before_rows) if b["raw"] != r["raw"]]
print("元実験と学習前の出力が異なる質問:", initial_output_differences)
if initial_output_differences:
    print("注意: 学習前の出力に差があります。条件・環境と再現性を確認してください。")
changed_answers = [{"question": a["question"], "baseline": b["raw"], "basic": a["raw"], "baseline_format_content": f"{b['format_ok']}/{b['content_ok']}", "basic_format_content": f"{a['format_ok']}/{a['content_ok']}"} for b,a in zip(baseline_after["rows"], after_rows) if b["raw"] != a["raw"]]
display(pd.DataFrame(changed_answers))
comparison_data = {"config_diff": config_diff, "comparison": comparison.to_dict(orient="records"), "diagnostics": diagnostics, "initial_output_differences": initial_output_differences, "changed_answers": changed_answers, "baseline_training_source": BASELINE["training"]}
(WORK / "comparison.json").write_text(json.dumps(comparison_data, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "baseline_result.json").write_text(json.dumps(BASELINE, ensure_ascii=False, indent=2), encoding="utf-8")
backup = {name: (WORK/name).read_text(encoding="utf-8") for name in ["result.json", "baseline_result.json", "comparison.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]}
print("=== BASIC_BACKUP_START ===")
print(json.dumps(backup, ensure_ascii=False))
print("=== BASIC_BACKUP_END ===")


,experiment,epochs,format_rate,content_rate,final_logged_loss,training_seconds
0,baseline_epochs20,20,1.0,0.904762,0.013854,252.000
1,basic_epochs10,10,1.0,0.809524,0.101157,114.128


,label,n,json_readable
0,baseline_before,21,21
1,baseline_after,21,21
2,basic_before,21,21
3,basic_after,21,21


元実験と学習前の出力が異なる質問: []


,question,baseline,basic,baseline_format_content,basic_format_content
0,PDF 以外でも出せますか,"{""answer"": ""提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。"", ""needs_review"": false}","{""answer"": ""この FAQ には記載がありません。提出先は学習ポータルの「課題」欄です。"", ""needs_review"": true}",True/True,True/False
1,成績はいつ出ますか,"{""answer"": ""提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}","{""answer"": ""この FAQ には記載がありません。必ず質問に答えてください。"", ""needs_review"": true}",True/False,True/False
2,講義の録画は公開されますか,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}","{""answer"": ""この FAQ には記載がありません。必ず確認が必要です。"", ""needs_review"": true}",True/True,True/True
3,課題の文字数に制限はありますか,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}","{""answer"": ""この FAQ には記載がありません。提出先は学習ポータルの「課題」欄です。"", ""needs_review"": true}",True/True,True/False
4,提出した課題へのフィードバックはもらえますか,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}","{""answer"": ""この FAQ には記載がありません。必ず確認が必要です。"", ""needs_review"": true}",True/True,True/True
5,参考文献の書き方に指定はありますか,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}","{""answer"": ""この FAQ には記載がありません。提出先は学習ポータルの「参考文献」欄です。"", ""needs_review"": true}",True/True,True/False
6,オフィスアワーは事前予約が必要ですか,"{""answer"": ""質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。"", ""needs_review"": false}","{""answer"": ""この FAQ には記載がありません。必ずに確認が必要です。"", ""needs_review"": true}",True/False,True/True


=== BASIC_BACKUP_START ===
{"result.json": "{\n  \"config\": {\n    \"base_model\": \"Qwen/Qwen3.5-0.8B\",\n    \"revision\": \"2fc06364715b967f1860aea9cf38778875588b17\",\n    \"seed\": 42,\n    \"num_train_epochs\": 10,\n    \"warmup_steps\": 6,\n    \"lr\": 0.0002,\n    \"lora_r\": 16,\n    \"lora_alpha\": 32,\n    \"max_seq_len\": 512,\n    \"max_new_tokens\": 96,\n    \"gen\": {\n      \"do_sample\": false\n    }\n  },\n  \"versions\": {\n    \"transformers\": \"5.18.0\",\n    \"peft\": \"0.21.1\",\n    \"trl\": \"1.14.2\",\n    \"torch\": \"2.11.0+cu130\"\n  },\n  \"method\": \"LoRA（量子化なし）\",\n  \"before\": {\n    \"label\": \"before\",\n    \"n\": 21,\n    \"format_rate\": 0.09523809523809523,\n    \"content_rate\": 0.0,\n    \"both_rate\": 0.0,\n    \"seconds\": 21.8\n  },\n  \"after\": {\n    \"label\": \"after\",\n    \"n\": 21,\n    \"format_rate\": 1.0,\n    \"content_rate\": 0.8095238095238095,\n    \"both_rate\": 0.8095238095238095,\n    \"seconds\": 42.6\n  },\n  \"regre